# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, subprocess

if not os.path.isdir("Flyranks-internship-assignmnet-1"):
    subprocess.run(["git", "clone", "--depth", "1",
        "https://github.com/MaryamNaveed-bioinfo/Flyranks-internship-assignmnet-1"], check=True)
os.chdir("Flyranks-internship-assignmnet-1")
print("Working dir:", os.getcwd())

subprocess.run(["pip", "install", "-q", "duckdb", "scikit-learn"], check=True)

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
print("Connected. Ready to query:", REL)

Working dir: /content/Flyranks-internship-assignmnet-1
Connected. Ready to query: hf://datasets/FlyRank/internship-warehouse


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Question shape:** "What drives CTR, and can we predict it better than a simple
position-bucket average?" This is a **"what drives X" regression question** the
target (`ctr`) is continuous, not a yes/no label.

**Method:** Random Forest Regressor + permutation importance, per the skill's guidance
for this question shape ("simple model + permutation importance"). Random Forest is a
reasonable step up from the Week-4 rule because it can use multiple features jointly
(not just one bucketed signal) and capture non-linear interactions, while still being
interpretable via feature importance unlike a black-box model, we can see and sanity
-check what it's actually leaning on.

**Baseline it must beat:** the Week-4 rule's `expected_ctr` the average CTR for a
page's position bucket. That baseline uses exactly ONE signal (position, bucketed into
4 groups). The model gets more signal: continuous position, impressions, and
engagement/channel features so if it can't beat a single-bucket average with all that
extra information, that's a real and honest finding, not a failure to hide.

**Metric:** Mean Absolute Error (MAE) between predicted CTR and actual CTR directly
comparable to the baseline's error (`ctr_gap`, which Week 4 already computed as
`expected_ctr − ctr`).

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped by `client_hash_id`**, not a random row split.

Per the `flyrank-data` skill: IDs are pseudonyms for grouping/splitting only, never
features. A random split would let the model see other pages from the same client in
training and be tested on more pages from that *same* client it could learn
client-specific quirks rather than a generalizable CTR pattern, inflating the score
without real predictive power.

Using `GroupShuffleSplit` on `client_hash_id`: entire clients are held out for testing,
so the model must generalize to clients it has never seen matching the real-world use
case, since FlyRank needs this to work for new clients.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Approach:** Compute the baseline prediction (position-bucket average CTR, same as
Week 4) on this dataset, then train a Random Forest Regressor on richer features using
a client-grouped split, and compare both on MAE against the same held-out test rows.
Iterating on `fact_content_daily_performance_sample.parquet` per the data skill's
guidance.

In [3]:
q_features = f"""
WITH page_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_pageviews) AS ga4_pageviews,
        SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic,
        SUM(sessions_direct) AS sessions_direct,
        SUM(sessions_referral) AS sessions_referral,
        SUM(sessions_social) AS sessions_social,
        SUM(sessions_paid) AS sessions_paid,
        SUM(sessions_ai) AS sessions_ai,
        SUM(scroll_events) AS scroll_events
    FROM read_parquet('{REL}/fact_content_daily_performance_sample.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    *,
    gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr
FROM page_agg
WHERE gsc_impressions >= 100
"""
df = con.sql(q_features).df()
print(f"Rows: {len(df)}, Clients: {df['client_hash_id'].nunique()}")
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101917, Clients: 46


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,scroll_events,ctr
0,client_3ffa76342f366962,content_bf89a688c0ec7cd7,103.0,0.0,70.131074,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
1,client_3ffa76342f366962,content_b9b011826cf38a50,1332.0,9.0,6.782582,12.0,10.0,0.0,0.0,16.0,0.0,0.0,0.0,0.0,0.0,8.0,0.006757
2,client_3ffa76342f366962,content_27ab46e9a1e75f45,161.0,0.0,9.363328,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000
3,client_3ffa76342f366962,content_1e3fe83db5900a14,471.0,7.0,18.204529,13.0,11.0,1.0,50.0,12.0,1.0,1.0,0.0,0.0,0.0,1.0,0.014862
4,client_3ffa76342f366962,content_1ff453e053447468,626.0,1.0,4.779055,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.001597


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [4]:
# Baseline: position-bucket average CTR (same logic as Week 4)
import pandas as pd

df['position_bucket'] = pd.cut(
    df['gsc_avg_position'],
    bins=[-0.01, 3, 10, 20, float('inf')],  # -0.01 so position=0 is included
    labels=['1-3 (top)', '4-10', '11-20', '21+']
)

bucket_avg = df.groupby('position_bucket', observed=True)['ctr'].mean().rename('baseline_pred')
df = df.merge(bucket_avg, on='position_bucket', how='left')

print("NaN baseline_pred:", df['baseline_pred'].isna().sum())
print(df[['gsc_avg_position', 'position_bucket', 'ctr', 'baseline_pred']].head())

NaN baseline_pred: 0
   gsc_avg_position position_bucket       ctr  baseline_pred
0         70.131074             21+  0.000000       0.002358
1          6.782582            4-10  0.006757       0.005314
2          9.363328            4-10  0.000000       0.005314
3         18.204529           11-20  0.014862       0.004555
4          4.779055            4-10  0.001597       0.005314


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

feature_cols = [
    'gsc_avg_position', 'gsc_impressions',
    'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
    'sessions_organic', 'sessions_direct', 'sessions_referral',
    'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events'
]

X = df[feature_cols]
y = df['ctr']
groups = df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = df['baseline_pred'].iloc[test_idx]

print(f"Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Train clients: {groups.iloc[train_idx].nunique()}, Test clients: {groups.iloc[test_idx].nunique()}")

Train rows: 69632, Test rows: 32285
Train clients: 36, Test clients: 10


In [6]:
model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_preds = model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_test)
model_mae = mean_absolute_error(y_test, model_preds)
base_rate = y_test.mean()

print(f"Baseline MAE: {baseline_mae:.6f}")
print(f"Random Forest MAE: {model_mae:.6f}")
print(f"Base rate: {base_rate:.6f}")

Baseline MAE: 0.003729
Random Forest MAE: 0.002850
Base rate: 0.005400


In [7]:
print(df[['gsc_clicks', 'sessions_organic', 'gsc_impressions', 'ctr']].corr())

                  gsc_clicks  sessions_organic  gsc_impressions       ctr
gsc_clicks          1.000000          0.987718         0.210049  0.405372
sessions_organic    0.987718          1.000000         0.254669  0.472815
gsc_impressions     0.210049          0.254669         1.000000  0.065722
ctr                 0.405372          0.472815         0.065722  1.000000


In [8]:
# sessions_organic correlates 0.988 with gsc_clicks — near-duplicate of the target.
# Drop it and retrain for an honest comparison.
feature_cols_clean = [c for c in feature_cols if c != 'sessions_organic']

X_train_clean = X_train[feature_cols_clean]
X_test_clean = X_test[feature_cols_clean]

model_clean = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
model_clean.fit(X_train_clean, y_train)
model_clean_preds = model_clean.predict(X_test_clean)

model_clean_mae = mean_absolute_error(y_test, model_clean_preds)

final_comparison = pd.DataFrame({
    'Method': [
        'Baseline (position-bucket avg)',
        'Random Forest (with sessions_organic — leaky)',
        'Random Forest (clean, sessions_organic removed)'
    ],
    'MAE': [baseline_mae, model_mae, model_clean_mae],
    'Base rate (avg actual CTR)': [base_rate, base_rate, base_rate]
})
print(final_comparison)

                                            Method       MAE  \
0                   Baseline (position-bucket avg)  0.003729   
1    Random Forest (with sessions_organic — leaky)  0.002850   
2  Random Forest (clean, sessions_organic removed)  0.003342   

   Base rate (avg actual CTR)  
0                      0.0054  
1                      0.0054  
2                      0.0054  


In [9]:
print("Rows with any NaN in GA4/session features:", X_test_clean.isna().any(axis=1).sum())
print("Total test rows:", len(X_test_clean))

Rows with any NaN in GA4/session features: 4857
Total test rows: 32285


**Result:** Random Forest achieves MAE 0.002853 vs. the baseline's 0.003729 when
including `sessions_organic` but that feature correlates 0.988 with `gsc_clicks`,
making it a near-duplicate of the target rather than a real driver. After removing it,
the honest result is: **Random Forest MAE 0.003342 vs. baseline MAE 0.003729 a
genuine ~10.4% error reduction**, on the same client-grouped test split (32,285 rows,
10 held-out clients never seen in training).

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on**

After removing the leaky `sessions_organic` feature, permutation importance shows
1. `gsc_impressions` (0.338) by far the strongest, and structurally sensible: CTR's
   denominator directly shapes how stable/noisy the ratio is.
2. `ga4_pageviews` (0.265)  a real, independent engagement signal, not a duplicate of
   GSC clicks.
3. `gsc_avg_position` (0.125) matches the CONFIRMED signal validated in Week 4.

Several features (`sessions_direct`, `sessions_ai`, `sessions_referral`) show *negative*
importance shuffling them slightly improved the model, meaning they add noise rather
than signal in this setup.

**Where the model is most wrong**

The 3 largest individual errors all share the same pattern: completely missing GA4 data
(every GA4/session column is NaN). Checking this systematically: **4,857 of 32,285 test
rows (≈15%)** have no GA4 tracking at all. For these pages the model has almost nothing
beyond position and impressions to work with, and badly underestimates CTR the worst
3 cases predicted ~0.003–0.004 vs. actual 0.05–0.07, roughly 15-20x off.

**Honest takeaway:** the model gives a real, modest improvement (~10.4% MAE reduction)
over the baseline, driven mainly by impressions and GA4 pageviews rather than the
near-duplicate `sessions_organic` feature. Its clearest blind spot is the ~15% of pages
with no GA4 coverage a `has_ga4_data` flag (per the data skill's guidance on
missingness) would let the model explicitly account for this gap rather than silently
guessing, and is a natural next improvement.

In [10]:
from sklearn.inspection import permutation_importance

perm_clean = permutation_importance(model_clean, X_test_clean, y_test, n_repeats=10, random_state=42, n_jobs=-1)

importance_clean_df = pd.DataFrame({
    'feature': feature_cols_clean,
    'importance_mean': perm_clean.importances_mean
}).sort_values('importance_mean', ascending=False)

print(importance_clean_df)

                     feature  importance_mean
1            gsc_impressions         0.346842
2              ga4_pageviews         0.271264
0           gsc_avg_position         0.124015
11             scroll_events         0.019294
5   ga4_total_engagement_sec         0.017211
3               ga4_sessions         0.014557
9              sessions_paid         0.000334
4       ga4_engaged_sessions        -0.004944
8            sessions_social        -0.018442
6            sessions_direct        -0.037003
10               sessions_ai        -0.064518
7          sessions_referral        -0.069817


In [11]:
# 3 concrete wrong cases
results = X_test_clean.copy()
results['actual_ctr'] = y_test.values
results['predicted_ctr'] = model_clean_preds
results['abs_error'] = (results['actual_ctr'] - results['predicted_ctr']).abs()

print(results.sort_values('abs_error', ascending=False).head(3))

        gsc_avg_position  gsc_impressions  ga4_pageviews  ga4_sessions  \
78661           4.325029          19728.0            NaN           NaN   
101913          0.213115            122.0            NaN           NaN   
27445          20.915508            219.0            NaN           NaN   

        ga4_engaged_sessions  ga4_total_engagement_sec  sessions_direct  \
78661                    NaN                       NaN              NaN   
101913                   NaN                       NaN              NaN   
27445                    NaN                       NaN              NaN   

        sessions_referral  sessions_social  sessions_paid  sessions_ai  \
78661                 NaN              NaN            NaN          NaN   
101913                NaN              NaN            NaN          NaN   
27445                 NaN              NaN            NaN          NaN   

        scroll_events  actual_ctr  predicted_ctr  abs_error  
78661             NaN    0.072993       0.0

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.